In [1]:
import os
import sys
import time
import json
from dotenv import load_dotenv

load_dotenv()
import logging

# logging.basicConfig(level=logging.DEBUG, stream=sys.stdout)
from pydantic import BaseModel
import importlib

import agents

importlib.reload(agents)

from agents.templates.play_zero_agent import PlayZeroAgent
from agents.structs import FrameData, GameState

import textwrap

def print_wrapped_text(text: str, width: int = 80):
    """
    Print the given text with word-wrapped lines for better readability in the terminal.

    Args:
        text (str): The input text to be printed.
        width (int): The maximum line width before wrapping. Default is 80.
    """
    wrapper = textwrap.TextWrapper(width=width)
    paragraphs = text.strip().split("\n\n")

    for paragraph in paragraphs:
        wrapped = wrapper.fill(paragraph)
        print(wrapped + "\n")

#  Agent.__init__() missing 5 required positional arguments: 'card_id', 'game_id', 'agent_name', 'ROOT_URL', and 'record'
play_zero_agent: PlayZeroAgent = PlayZeroAgent(
    card_id="play_zero_agent",
    game_id="play_zero_agent",
    agent_name="PlayZeroAgent",
    ROOT_URL="http://localhost:8000",
    record=False,
)

runs = [
    {
        "game_id": "ls20",
        "level": 1,
        "video_path": "/workspaces/ARC-AGI-3-Agents/recordings/game_analysis_ls20-f340c8e5138e_track1.mp4",
        "scorecard_file_path": "/workspaces/ARC-AGI-3-Agents/recordings/ls20-f340c8e5138e.playzeroagent.gemini-2.5-flash.with-observe.gemini-2.5-flash.8aefc1ea-8e65-41ec-9272-a8faff24ddb1.recording.jsonl",
        "frame_start": 0,
        "frame_end": 55,
        "expected_goal": "You need to move the \"Orange-Capped Blue Block (6x7)\" to the target \"8x7Grid_BlackHead_BlueEye_WhiteSnout\"",
    },
    {
        "game_id": "ls20",
        "level": 1,
        "video_path": "/workspaces/ARC-AGI-3-Agents/recordings/game_analysis_ls20-f340c8e5138e_track2.mp4",
        "scorecard_file_path": "/workspaces/ARC-AGI-3-Agents/recordings/ls20-f340c8e5138e.playzeroagent.gemini-2.5-flash.with-observe.gemini-2.5-flash.fa1f97de-edb6-47c7-98bb-eff9f689d487.recording.jsonl",  
        "frame_start": 0,
        "frame_end": 55,
        "expected_goal": "You need to move the \"Orange-Capped Blue Block (6x7)\" to the target \"8x7Grid_BlackHead_BlueEye_WhiteSnout\"",
    },
    {
        "game_id": "vc33",
        "level": 1,
        "video_path": "/workspaces/ARC-AGI-3-Agents/recordings/game_analysis_vc33-58ec4396715d_track1.mp4",
        "scorecard_file_path": "recordings/vc33-58ec4396715d.playzeroagent.gemini-2.5-flash.with-observe.gemini-2.5-flash.5fe87248-8898-45b1-8634-84294454be48.recording.jsonl",
        "frame_start": 0,
        "frame_end": 55,
        "expected_goal": "Click red button and blue button and notice the effect",
    },
    {
        "game_id": "vc33",
        "level": 1,
        "video_path": "",
        "scorecard_file_path": "recordings/vc33-58ec4396715d.playzeroagent.gemini-2.5-flash.with-observe.gemini-2.5-flash.28b07367-701c-470f-93d6-b302ecbbc733.recording.jsonl",
        "frame_start": 0,
        "frame_end": 55,
        "expected_goal": "Click red button and blue button and notice the effect",
    }
]

def get_frames(scorecard_file_path):
    with open(scorecard_file_path, "r") as file:
        grid_jsons = [json.loads(line) for line in file]
    frames = [FrameData(**frame_json["data"]) for frame_json in grid_jsons]
    return frames

def write_eval_log(evaluation_result):
    with open("eval.log", "a") as eval_log_file:
        print_wrapped_text(f"Evaluation Result:\n\n {evaluation_result}\n")
        eval_log_file.write(f"Evaluation Result:\n\n {evaluation_result}\n")

# Eval prompt for multiple_hypothesis_text

EVAL_PROMPT = """Give score and reason of whether the multiple hypothesis can be used to generate the expected goal

Expected Goal: <expected_goal>{expected_goal}</expected_goal>

Multiple Hypothesis Text: <multiple_hypothesis_text>{multiple_hypothesis_text}</multiple_hypothesis_text>

Example output json:
```json
{{
    "reason": "<max of 100 words>",
    "score": "<float score from 0.0 to 1.0>"
}}
```
"""

GOAL_RELEVANCE_PROMPT = """Give score and reason of whether the generated goal can be used to achieve the expected goal

Expected Goal: <expected_goal>{expected_goal}</expected_goal>

Generated Goal: <generated_goal>{generated_goal}</generated_goal>

Example output json:
```json
{{
    "reason": "<max of 100 words>",
    "score": "<float score from 0.0 to 1.0>"
}}
```
"""


def evaluate_multiple_hypothesis_text(multiple_hypothesis_text: str, expected_goal: str):
    prompt = EVAL_PROMPT.format(
        expected_goal=expected_goal,
        multiple_hypothesis_text=multiple_hypothesis_text
    )
    response = play_zero_agent.client.chat.completions.create(
        model="gemini-2.5-flash",
        messages = [
            {
                "role": "user",
                "content": prompt,
            }
        ]
    )
    json_text = response.choices[0].message.content.strip()
    json_text = play_zero_agent.extract_first_json_block(json_text)
    json_data = json.loads(json_text)
    return json_data

def eval_goal_relevance(generated_goal: str, expected_goal: str):
    prompt = GOAL_RELEVANCE_PROMPT.format(
        expected_goal=expected_goal,
        generated_goal=generated_goal
    )
    response = play_zero_agent.client.chat.completions.create(
        model="gemini-2.5-flash",
        messages = [
            {
                "role": "user",
                "content": prompt,
            }
        ]
    )
    json_text = response.choices[0].message.content.strip()
    json_text = play_zero_agent.extract_first_json_block(json_text)
    json_data = json.loads(json_text)
    return json_data

class RunResult(BaseModel):
    logical_analysis_actions_summary: str = ""
    eval_multiple_hypothesis_result: dict = {}
    eval_goal_relevance_result: dict = {}
    multiple_hypothesis_text: str = ""
    goal: str = ""
    elements_text: str = ""

def store_run_results(run_results: list[dict]):
    # generate a unique path for the run results file
    timestamp = time.strftime("%Y%m%d-%H%M%S")
    run_results_path = f"data/run_results_{timestamp}.json"
    with open(run_results_path, "w") as file:
        json.dump(run_results, file)


/workspaces/ARC-AGI-3-Agents/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

# def test_run(run) -> RunResult:
#     video_path = run["video_path"]
#     scorecard_file_path = run["scorecard_file_path"]
#     frame_start = run.get("frame_start", 0)
#     frame_end = run.get("frame_end", None)
#     expected_goal = run.get("expected_goal", "")

#     frames = get_frames(scorecard_file_path)
#     frames_for_analysis = frames[frame_start:frame_end] if frame_end else frames[frame_start:]

#     print(f"Running analysis for video: {video_path}")
#     print(f"Scorecard file: {scorecard_file_path}")
#     print(f"Frames from {frame_start} to {frame_end if frame_end else 'end'}")

#     logical_analysis_actions_summary = play_zero_agent.generate_logical_analysis_summary(frames_for_analysis)
#     print(f"Logical Analysis Actions Summary:\n\n {logical_analysis_actions_summary}")
#     elements_text = play_zero_agent.generate_element_titles_from_video(
#         video_file_path=video_path,
#     )
#     print_wrapped_text(f"elements_text:\n\n {elements_text}")

#     multiple_hypothesis_text = play_zero_agent.generate_multiple_random_hypothesis_from_video(
#         video_file_path=video_path,
#         logical_analysis_actions_summary=logical_analysis_actions_summary,
#         elements_text=elements_text,
#     )
#     print_wrapped_text(f"Multiple Hypothesis Text:\n\n {multiple_hypothesis_text}")
#     eval_multiple_hypothesis_result = evaluate_multiple_hypothesis_text(
#         multiple_hypothesis_text=multiple_hypothesis_text,
#         expected_goal=expected_goal,
#     )
#     write_eval_log(eval_multiple_hypothesis_result)
#     goal = play_zero_agent.generate_top_hypothesis(
#         multiple_hypothesis_text=multiple_hypothesis_text,
#         logical_analysis_actions_summary=logical_analysis_actions_summary,
#     )
#     print_wrapped_text(f"Generated Goal:\n\n {goal}")
#     eval_goal_relevance_result = eval_goal_relevance(
#         generated_goal=goal,
#         expected_goal=expected_goal
#     )
#     write_eval_log(eval_goal_relevance_result)


#     return RunResult(
#         logical_analysis_actions_summary=logical_analysis_actions_summary,
#         multiple_hypothesis_text=multiple_hypothesis_text,
#         eval_multiple_hypothesis_result=eval_multiple_hypothesis_result,
#         eval_goal_relevance_result=eval_goal_relevance_result,
#         goal=goal,
#         elements_text=elements_text
#     )

# run_results = []
# for run in runs:
#     run_result = test_run(run)
#     run_results.append({
#         "run": run,
#         "result": run_result.model_dump()
#     })
# store_run_results(run_results)

In [3]:
# run_result = run_results[0]
# # generate goal and evaluate it
# goal = play_zero_agent.generate_top_hypothesis(
#     multiple_hypothesis_text=run_result["result"]["multiple_hypothesis_text"],
#     logical_analysis_actions_summary=run_result["result"]["logical_analysis_actions_summary"],
# )
# print_wrapped_text(f"Generated Goal:\n\n {goal}")

# eval_goal_relevance_result = eval_goal_relevance(
#     generated_goal=goal,
#     expected_goal=run["expected_goal"]
# )
# write_eval_log(eval_goal_relevance_result)

# print_wrapped_text(f"Eval Goal Relevance Result:\n\n {eval_goal_relevance_result}")

In [4]:
# from agents.templates.play_zero_agent import TOP_HYPOTHESIS_RETRIEVER_PROMPT


# prompt = TOP_HYPOTHESIS_RETRIEVER_PROMPT.format(
#     multiple_hypothesis_text=run_result["result"]["multiple_hypothesis_text"],
#     logical_analysis_actions_summary=run_result["result"]["logical_analysis_actions_summary"],
# )
# print_wrapped_text(f"Top Hypothesis Retriever Prompt:\n\n {prompt}")

In [ ]:
from google.genai import types

FRAME_EVENT_PROMPT = """{{"startFrame": {start_frame}, "endFrame": {end_frame}, "action_taken": "{action_taken}", "score": {score}, "effect": "{effect}"}}"""
EVENT_CHAIN_FILLER_MODEL = "gemini-2.5-pro"
EVENT_CHAIN_FILLER_PROMPT = """This video is game play with the below actions (WASD and click) taken on unknown game.

The game is designed based on below Constraints
- Easy for humans (can pick it up in <1 min of game play)
- Core Knowledge Priors (no language, trivia, cultural symbols)
- Should require no instructions to play
- Should be fun for humans and playable in 5-10 minutes
- Innovative and novel game mechanics encouraged (Hidden state, theory of mind, long term planning, navigating other agents, etc.)
- If a level is cleared, the score will be increased

Here are the actions that your player can take
W: Move Up
A: Move Left
S: Move Down
D: Move Right
CLICK(x,y): Click on the area by giving x,y space (x: <0, 63>, y: <0, 63>)

Actions taken in this game play
{event_chain_text}

You need to fill the event action chain with the game effect that you in the game play based on the respective frames
"""
logger = logging.getLogger(__name__)
def generate_event_chain(self: PlayZeroAgent, effective_frames: list[FrameData]) -> str:
    count = 0
    event_chain = []
    prev_frame = effective_frames[0]
    for frame in effective_frames:
        action_text = frame.action_input.reasoning["previous_action_text"]
        frame_count = len(frame.frame)
        game_action = self.convert_action_text_to_game_action(action_text)
        effect = ""
        if game_action.is_complex():
            x = game_action.action_data.x
            y = game_action.action_data.y
        
            cell_value = prev_frame.frame[-1][y][x]
            color = self.get_color_for_cell_value(cell_value)
            if color:
                effect = f"Clicking {color} cell"
        frame_event = FRAME_EVENT_PROMPT.format(start_frame=count, end_frame=count + frame_count, score=frame.score, action_taken=action_text, effect=effect)
        event_chain.append(frame_event)
        count += frame_count
    return "".join(event_chain)


def fill_missing_data_in_event_chain_using_video(
    self,
    video_file_path: str,
    event_chain_text: str,
) -> str:
        """Filling event chain from a video file."""
        logger.info(f"Filling event chain from video: {video_file_path}")
        if not os.path.exists(video_file_path):
            logger.error(f"Video file does not exist: {video_file_path}")
            return "Analysis not yet done."
        video_bytes = open(video_file_path, 'rb').read()

        event_chain_text_response = self.generate_content_using_gemini(
            model=EVENT_CHAIN_FILLER_MODEL,
            contents=types.Content(
                parts=[
                    types.Part(
                        inline_data=types.Blob(data=video_bytes, mime_type='video/mp4')
                    ),
                    types.Part(text=EVENT_CHAIN_FILLER_PROMPT.format(
                        event_chain_text=event_chain_text,
                    ))
                ]
            )
        )
        
        filled_event_chain_text = event_chain_text_response.text.strip()
        self.track_tokens(
            event_chain_text_response.usage_metadata.total_token_count, event_chain_text_response.text
        )
        logger.info(f"Event Chain filled: {filled_event_chain_text}")
        return filled_event_chain_text

frames = get_frames(runs[3]["scorecard_file_path"])[:-1]
effective_frames = play_zero_agent.generate_video_from_grids(frames, "output.mp4", fps=1, skip_repeated_frames=True)
event_chain_text = generate_event_chain(play_zero_agent, effective_frames=effective_frames)
filled_event_chain_text = fill_missing_data_in_event_chain_using_video(play_zero_agent, "output.mp4", event_chain_text)
print_wrapped_text(f"Event Chain:\n\n {event_chain_text}", width=90)
print(f"Filled Event Chain:\n\n {filled_event_chain_text}")
